# Lesson 08 — From Graph Bottlenecks to Value

## Goal

Use graph analysis to identify high-value AI interventions by modelling workflows as
directed weighted graphs. Find hidden bottleneck nodes, critical edges, and single
points of failure that traditional time-series analysis misses.

---

## Learning Objectives

By the end of this lesson, you will:

1. **Build process graphs** — Convert handoff data into directed weighted graphs
2. **Apply centrality metrics** — Rank bottleneck nodes by betweenness and degree centrality
3. **Find critical edges** — Identify handoffs with the highest friction score (cases x wait_hours)
4. **Answer six graph questions** — Hidden bottleneck, slowest path, single point of failure, etc.
5. **Connect to L06 friction cost** — Attach EUR values to each bottleneck node
6. **Prioritise AI interventions** — Rank automations by graph evidence + financial impact

---

## Prerequisites

- **Lesson 06:** Cost of Friction (rework cost, expert interruption cost, friction %)
- **Lesson 07:** Value Stream Metrics (lead time decomposition, bottleneck identification)

---

## Core Insight

**High betweenness centrality = single point of failure.**
A node on many shortest paths will stall a large fraction of work when it becomes slow.
Combining centrality with L06 friction cost gives a ranked, financially grounded AI intervention list.

## Part 1 — Graph Concepts for Process Analysis

### Why Model Workflows as Graphs?

A process log tells you **when** a case was slow.
A graph tells you **why** — which person or approval sits at the centre of too many paths.

### Graph Vocabulary

| Term | Definition | Process Example |
|------|-----------|----------------|
| **Node** | An entity in the network | Person, team, system, approval step |
| **Directed edge** | A one-way relationship | Analyst *hands off to* Manager |
| **Edge weight** | Numerical attribute on an edge | # cases, avg wait hours, friction score |
| **Betweenness centrality** | Fraction of shortest paths passing through a node | Manager on 55% of invoice paths |
| **Degree centrality** | Normalised count of direct connections | Step touched by 4 different roles |
| **Friction score** | cases x avg_wait_hours per edge | 120 cases x 30 h = 3,600 |

### Six Graph Questions That Reveal AI Opportunities

1. **Who is the hidden bottleneck?** -> highest betweenness centrality node
2. **Which approval path is slowest?** -> edge with highest avg_wait_hours
3. **Which step is central to many workflows?** -> high degree node
4. **Which system creates manual bridges?** -> node with self-loops or rework edges
5. **Which team receives incomplete requests?** -> node with high in-degree from rework paths
6. **Which group depends on one expert?** -> star topology around one high-centrality node

### Connection to L06 and L07

- **L06** measured *how much* friction costs (EUR 308 invoice, EUR 9,603 support, EUR 4,022 reporting)
- **L07** measured *where* friction lives in time (queues, revision cycles, specialist waits)
- **L08** reveals *why* it persists structurally -- which nodes are unavoidable chokepoints

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

print('Libraries loaded')
print('  networkx:', nx.__version__)
print()
print('Ready to build process graphs')

---

## Part 2 — Core Graph Utilities

Three reusable functions:
- `build_process_graph(handoffs_df)` -- turns a handoff DataFrame into a weighted DiGraph
- `top_bottleneck_nodes(G)` -- ranks nodes by betweenness centrality
- `top_friction_edges(G)` -- ranks edges by friction score (cases x avg_wait_hours)

In [ ]:
def build_process_graph(handoffs_df):
    G = nx.DiGraph()
    for _, row in handoffs_df.iterrows():
        G.add_edge(
            row['from_role'],
            row['to_role'],
            weight=row['cases'],
            avg_wait_hours=row['avg_wait_hours'],
            friction_score=row['cases'] * row['avg_wait_hours'],
        )
    return G


def top_bottleneck_nodes(G, top_n=5):
    betweenness = nx.betweenness_centrality(G, weight='avg_wait_hours', normalized=True)
    degree = nx.degree_centrality(G)
    rows = []
    for node in G.nodes():
        in_weight = sum(d['weight'] for _, _, d in G.in_edges(node, data=True))
        rows.append({
            'node': node,
            'betweenness': betweenness[node],
            'degree_centrality': degree[node],
            'in_degree': G.in_degree(node),
            'out_degree': G.out_degree(node),
            'total_cases_in': in_weight,
        })
    df = pd.DataFrame(rows).sort_values('betweenness', ascending=False).reset_index(drop=True)
    return df.head(top_n)


def top_friction_edges(G, top_n=5):
    rows = [
        {'from': u, 'to': v, 'cases': d['weight'],
         'avg_wait_hours': d['avg_wait_hours'], 'friction_score': d['friction_score']}
        for u, v, d in G.edges(data=True)
    ]
    return (
        pd.DataFrame(rows)
        .sort_values('friction_score', ascending=False)
        .reset_index(drop=True)
        .head(top_n)
    )


print('Functions loaded:')
print('  build_process_graph(handoffs_df)')
print('  top_bottleneck_nodes(G, top_n=5)')
print('  top_friction_edges(G, top_n=5)')

---

## Part 3 — Worked Example 1: Invoice Approval Graph

### Context from L06 / L07

- **4,200 invoices/year** (1,050 in Q1)
- L06 friction: **EUR 308** (6% of total cost) -- rework EUR 231, manager escalation EUR 77
- L07 lead time: **1.3 days average**, 99% waiting in queues
- L07 bottleneck: manager approval queue (20 h average wait)

**L08 question:** Is Manager a structural bottleneck, or just occasionally slow?
Build the approval handoff graph and check centrality.

In [ ]:
invoice_handoffs = pd.DataFrame({
    'from_role':      ['Clerk',   'Analyst', 'Analyst', 'Clerk',  'Manager'],
    'to_role':        ['Analyst', 'Analyst', 'Manager', 'Analyst','Accounting'],
    'cases':          [1050,       1050,       21,        53,       21],
    'avg_wait_hours': [24,         1,          20,        24,       0.5],
    'description':    [
        'Initial submission to analyst queue',
        'Analyst routine approval',
        'Escalation to manager (2%)',
        'Rework resubmission (5%)',
        'Manager passes to accounting',
    ],
})
invoice_handoffs['friction_score'] = invoice_handoffs['cases'] * invoice_handoffs['avg_wait_hours']
G_invoice = build_process_graph(invoice_handoffs)
print('INVOICE APPROVAL -- Handoff Graph')
print('=' * 70)
print(f'  Nodes: {list(G_invoice.nodes())}')
print(f'  Edges: {G_invoice.number_of_edges()}')
print()
print(invoice_handoffs[['from_role','to_role','cases','avg_wait_hours','friction_score']].to_string(index=False))

In [ ]:
invoice_bottlenecks = top_bottleneck_nodes(G_invoice)
invoice_top_edges   = top_friction_edges(G_invoice)
print('INVOICE -- TOP BOTTLENECK NODES (by betweenness centrality)')
print('=' * 70)
print(invoice_bottlenecks.to_string(index=False))
print()
print('INVOICE -- TOP FRICTION EDGES (cases x avg_wait_hours)')
print('=' * 70)
print(invoice_top_edges.to_string(index=False))
print()
top_inv = invoice_bottlenecks.iloc[0]
print('KEY FINDING:')
print(f"  '{top_inv['node']}' has betweenness={top_inv['betweenness']:.3f}")
print('  -> This node sits on the most invoice case paths')
print()
print('L06 CONNECTION:')
print('  Manager escalation: 21 cases x 20 h = 420 friction units')
print('  L06 measured EUR 77 manager cost  -- graph confirms structural role')
print('  Rework resubmission: 53 cases x 24 h = 1,272 friction units (higher!)')
print('  L06 measured EUR 231 rework cost  -- rework loop is larger friction source')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
pos = nx.spring_layout(G_invoice, k=2.5, seed=42)
btw_inv = nx.betweenness_centrality(G_invoice, weight='avg_wait_hours', normalized=True)
node_colors = [btw_inv.get(n, 0) for n in G_invoice.nodes()]
node_sizes  = [nx.degree_centrality(G_invoice)[n] * 4000 + 600 for n in G_invoice.nodes()]
sc = nx.draw_networkx_nodes(G_invoice, pos, node_color=node_colors, node_size=node_sizes,
                             cmap='RdYlGn_r', vmin=0, vmax=0.7, ax=ax)
nx.draw_networkx_labels(G_invoice, pos, font_size=9, font_weight='bold', ax=ax)
edge_w = [G_invoice[u][v]['weight'] / 150 for u, v in G_invoice.edges()]
nx.draw_networkx_edges(G_invoice, pos, width=edge_w, alpha=0.6,
                       edge_color='#555', arrowsize=18,
                       connectionstyle='arc3,rad=0.1', ax=ax)
elabels = {(u,v): f"{d['cases']}c/{d['avg_wait_hours']:.0f}h" for u,v,d in G_invoice.edges(data=True)}
nx.draw_networkx_edge_labels(G_invoice, pos, elabels, font_size=7, ax=ax)
plt.colorbar(sc, ax=ax, label='Betweenness Centrality (red=bottleneck)')
ax.set_title('Invoice Approval -- Handoff Graph\nNode colour=betweenness | Node size=degree | Edge=cases/wait', fontsize=11, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()
print('Red nodes = structural bottlenecks.  Thick edges = high case volume.')

---

## Part 4 — Worked Example 2: Support Triage Escalation Graph

### Context from L06 / L07

- **18,000 tickets/year** (4,500 in Q1)
- L06 friction: **EUR 9,603** (27% of total cost) -- rework EUR 3,528, specialist escalation EUR 6,075
- L07 lead time: **2.1 days non-escalated vs 8.2 days escalated** (+6.1 day penalty)
- L07 bottleneck: specialist queue (8 h average wait for escalated cases)

**L08 question:** Is Specialist or Analyst the structural bottleneck?
Build the escalation dependency graph and check both.

In [ ]:
support_handoffs = pd.DataFrame({
    'from_role':      ['Customer', 'Analyst',  'Analyst',   'Analyst', 'Specialist'],
    'to_role':        ['Analyst',  'Analyst',  'Specialist','System',  'System'],
    'cases':          [4500,        360,         675,         3825,      675],
    'avg_wait_hours': [1,           8,           8,           2,         0.5],
    'description':    [
        'Customer submits ticket',
        'Misroute rework loop (8%)',
        'Escalation to specialist (15%)',
        'Non-escalated: analyst resolves',
        'Specialist closes ticket',
    ],
})
support_handoffs['friction_score'] = support_handoffs['cases'] * support_handoffs['avg_wait_hours']
G_support = build_process_graph(support_handoffs)
print('SUPPORT TRIAGE -- Handoff Graph')
print('=' * 70)
print(f'  Nodes: {list(G_support.nodes())}')
print(f'  Edges: {G_support.number_of_edges()}')
print()
print(support_handoffs[['from_role','to_role','cases','avg_wait_hours','friction_score']].to_string(index=False))

In [ ]:
support_bottlenecks = top_bottleneck_nodes(G_support)
support_top_edges   = top_friction_edges(G_support)
print('SUPPORT -- TOP BOTTLENECK NODES')
print('=' * 70)
print(support_bottlenecks.to_string(index=False))
print()
print('SUPPORT -- TOP FRICTION EDGES')
print('=' * 70)
print(support_top_edges.to_string(index=False))
print()
top_sup = support_bottlenecks.iloc[0]
print('KEY FINDING:')
print(f"  '{top_sup['node']}' is top bottleneck (betweenness={top_sup['betweenness']:.3f})")
print()
print('L06 CONNECTION:')
print('  Specialist escalation: 675 cases x 8 h = 5,400 friction units (highest edge)')
print('  L06 measured EUR 6,075 specialist cost  -- largest friction layer in support')
print('  Analyst rework loop:   360 cases x 8 h = 2,880 friction units')
print('  L06 measured EUR 3,528 rework cost')
print()
print('IMPLICATION:')
print('  LLM intent classifier addresses both edges:')
print('  (1) Cuts misroute 8% -> 2%: removes rework loop')
print('  (2) Cuts escalation 15% -> 5%: removes highest-friction edge')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
pos_s = nx.spring_layout(G_support, k=2.5, seed=7)
btw_sup = nx.betweenness_centrality(G_support, weight='avg_wait_hours', normalized=True)
node_colors_s = [btw_sup.get(n, 0) for n in G_support.nodes()]
node_sizes_s  = [nx.degree_centrality(G_support)[n] * 4000 + 600 for n in G_support.nodes()]
sc2 = nx.draw_networkx_nodes(G_support, pos_s, node_color=node_colors_s, node_size=node_sizes_s,
                              cmap='RdYlGn_r', vmin=0, vmax=0.7, ax=ax)
nx.draw_networkx_labels(G_support, pos_s, font_size=9, font_weight='bold', ax=ax)
edge_w_s = [G_support[u][v]['weight'] / 500 for u, v in G_support.edges()]
nx.draw_networkx_edges(G_support, pos_s, width=edge_w_s, alpha=0.6,
                       edge_color='#555', arrowsize=18,
                       connectionstyle='arc3,rad=0.1', ax=ax)
elabels_s = {(u,v): f"{d['cases']}c/{d['avg_wait_hours']:.0f}h" for u,v,d in G_support.edges(data=True)}
nx.draw_networkx_edge_labels(G_support, pos_s, elabels_s, font_size=7, ax=ax)
plt.colorbar(sc2, ax=ax, label='Betweenness Centrality (red=bottleneck)')
ax.set_title('Support Triage -- Escalation Dependency Graph\nNode colour=betweenness | Edge=cases/wait', fontsize=11, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()
print('Self-loop on Analyst = rework cycle.  Thick edge to Specialist = escalation path.')

---

## Part 5 — Worked Example 3: Management Reporting Review Graph

### Context from L06 / L07

- **52 reports/year** (13 in Q1)
- L06 friction: **EUR 4,022** (44% of total cost -- highest % of all 3 workflows)
- L07 lead time: **12 days (0 revisions) to 28 days (2+ revisions)**, high variance
- L07 bottleneck: executive review cycle (6 days per revision loop)

**L08 question:** Does the revision cycle create a structural dependency on Executive?
Build the review cycle graph and measure centrality.

In [ ]:
reporting_handoffs = pd.DataFrame({
    'from_role':      ['Analyst',  'Executive','Analyst',  'Executive','Executive'],
    'to_role':        ['Executive','Analyst',  'Executive','Analyst',  'Published'],
    'cases':          [13,          4,           4,          1,          13],
    'avg_wait_hours': [72,          48,          72,         48,         12],
    'description':    [
        'Initial draft to executive review',
        'Revision required -- back to analyst (30%)',
        'Revised draft to executive',
        'Second revision required (5%)',
        'Executive approves -- published',
    ],
})
reporting_handoffs['friction_score'] = reporting_handoffs['cases'] * reporting_handoffs['avg_wait_hours']
G_reporting = build_process_graph(reporting_handoffs)
print('MANAGEMENT REPORTING -- Handoff Graph')
print('=' * 70)
print(f'  Nodes: {list(G_reporting.nodes())}')
print(f'  Edges: {G_reporting.number_of_edges()}')
print()
print(reporting_handoffs[['from_role','to_role','cases','avg_wait_hours','friction_score']].to_string(index=False))

In [ ]:
reporting_bottlenecks = top_bottleneck_nodes(G_reporting)
reporting_top_edges   = top_friction_edges(G_reporting)
print('REPORTING -- TOP BOTTLENECK NODES')
print('=' * 70)
print(reporting_bottlenecks.to_string(index=False))
print()
print('REPORTING -- TOP FRICTION EDGES')
print('=' * 70)
print(reporting_top_edges.to_string(index=False))
print()
top_rep = reporting_bottlenecks.iloc[0]
print('KEY FINDING:')
print(f"  '{top_rep['node']}' has betweenness={top_rep['betweenness']:.3f}")
print('  -> ALL 13 reports must pass through Executive at least once')
print('  -> 72 h average wait = 3 days of lead time per report')
print()
print('L06 CONNECTION:')
print('  44% of reporting cost is friction -- highest of all 3 workflows')
print('  Graph confirms: Executive is unavoidable AND slow (72 h wait)')
print()
print('IMPLICATION:')
print('  LLM pre-review: cut revision rate 30% -> 10%  ==>  removes 2 revision-loop edges')
print('  LLM-assisted draft: tighten first submission  ==>  reduces 72 h wait at Executive')

---

## Part 6 — Cross-Workflow Bottleneck Comparison

### Ranking the Three Workflows for AI Investment

Combine L06 friction cost + L07 lead time + L08 graph centrality to build a prioritised intervention list.

In [ ]:
btw_inv_m = nx.betweenness_centrality(G_invoice,   weight='avg_wait_hours', normalized=True)
btw_sup_m = nx.betweenness_centrality(G_support,   weight='avg_wait_hours', normalized=True)
btw_rep_m = nx.betweenness_centrality(G_reporting, weight='avg_wait_hours', normalized=True)
top_inv_n = max(btw_inv_m, key=btw_inv_m.get)
top_sup_n = max(btw_sup_m, key=btw_sup_m.get)
top_rep_n = max(btw_rep_m, key=btw_rep_m.get)
comparison = pd.DataFrame({
    'Workflow':              ['Invoice',   'Support',   'Reporting'],
    'L06 Friction EUR':      [308,          9603,        4022],
    'L06 Friction %':        [6,            27,          44],
    'L07 Avg Lead Time (d)': [1.3,          3.5,         15.0],
    'Graph Bottleneck':      [top_inv_n,    top_sup_n,   top_rep_n],
    'Top Centrality':        [round(btw_inv_m[top_inv_n], 3),
                              round(btw_sup_m[top_sup_n], 3),
                              round(btw_rep_m[top_rep_n], 3)],
    'Recommended AI Action': [
        'Rule-based auto-approve (98% of cases)',
        'LLM classifier to cut escalation 15%->5%',
        'LLM pre-review to cut revision rate 30%->10%',
    ],
})
print('CROSS-WORKFLOW GRAPH COMPARISON')
print('=' * 110)
print(comparison.to_string(index=False))
print()
print('PRIORITISATION:')
print('  1. SUPPORT    -- EUR 9,603 friction + high centrality + clear LLM win')
print('  2. REPORTING  -- EUR 4,022 friction + executive bottleneck (100% centrality)')
print('  3. INVOICE    -- EUR 308 friction (small scale) -- prioritise at high volume')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
wf_names  = ['Invoice', 'Support', 'Reporting']
fric_eur  = [308, 9603, 4022]
centrals  = [btw_inv_m[top_inv_n], btw_sup_m[top_sup_n], btw_rep_m[top_rep_n]]
lead_days = [1.3, 3.5, 15.0]
colors_   = ['#3498db', '#e74c3c', '#f39c12']
for wf, fe, ce, lt, col in zip(wf_names, fric_eur, centrals, lead_days, colors_):
    ax.scatter(ce, fe, s=lt*80, color=col, alpha=0.75, edgecolors='black', linewidths=1.5, label=wf)
    ax.annotate(wf, (ce, fe), textcoords='offset points', xytext=(10, 5), fontsize=10, fontweight='bold')
ax.set_xlabel('Top Node Betweenness Centrality', fontsize=11, fontweight='bold')
ax.set_ylabel('L06 Friction Cost (EUR)', fontsize=11, fontweight='bold')
ax.set_title('Graph Centrality vs Friction Cost\n(bubble size = L07 avg lead time days)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('Upper-right quadrant = high friction EUR AND high centrality = highest AI priority')

---

## Part 7 — Exercises

### Exercise 1: Guided — Build and Analyse the Masterclass Sample Graph

The masterclass specification includes a sample approval graph with roles:
Analyst, Manager, Legal, CFO, and AP.

```
Analyst  -> Manager    120 cases   12 h wait
Manager  -> Legal       80 cases   30 h wait
Manager  -> CFO         60 cases   48 h wait
Legal    -> Manager     30 cases   20 h wait
Manager  -> AP         100 cases   10 h wait
CFO      -> AP          40 cases   72 h wait
```

1. Build the directed weighted graph using `build_process_graph()`
2. Print betweenness centrality for every node
3. Which node is the structural bottleneck?
4. Which edge has the highest friction score?
5. What AI action would most reduce Manager's centrality?

In [ ]:
# Exercise 1 -- ANSWER KEY
print('EXERCISE 1: Masterclass Sample Graph')
print('=' * 70)
print()
sample_handoffs = pd.DataFrame({
    'from_role':      ['Analyst', 'Manager', 'Manager', 'Legal',  'Manager', 'CFO'],
    'to_role':        ['Manager', 'Legal',   'CFO',     'Manager','AP',      'AP'],
    'cases':          [120,        80,         60,        30,       100,       40],
    'avg_wait_hours': [12,         30,         48,        20,       10,        72],
})
G_sample = build_process_graph(sample_handoffs)
sample_btw = nx.betweenness_centrality(G_sample, weight='avg_wait_hours', normalized=True)
print('Q2 -- Betweenness centrality per node:')
for node, score in sorted(sample_btw.items(), key=lambda x: x[1], reverse=True):
    print(f'  {node:10s}: {score:.4f}')
print()
top_s = max(sample_btw, key=sample_btw.get)
print(f'Q3 -- Structural bottleneck: {top_s} (centrality={sample_btw[top_s]:.4f})')
print('     Manager sits on most shortest paths -- automating it unblocks most cases')
print()
top_e = top_friction_edges(G_sample, top_n=1)
print('Q4 -- Highest friction edge:')
print(top_e.to_string(index=False))
print()
print('Q5 -- AI action to reduce Manager centrality:')
print('  Manager->CFO has highest wait (48 h) -- a high-value approval gate.')
print('  Action 1: Rule-based auto-approve CFO cases below amount threshold.')
print('  Action 2: LLM pre-screening to reduce Legal escalations (removes Legal->Manager loop).')
print('  Effect: Both reduce Manager betweenness by removing inbound/outbound edges.')

---

### Exercise 2: Open-Ended — Model the Expert Dependency Network

A company has three operational analysts who all depend on one senior data engineer for:

- Ad-hoc SQL queries: 30 requests/month, 4 h avg wait
- Dashboard fixes: 10 requests/month, 8 h avg wait
- Data pipeline debugging: 5 requests/month, 24 h avg wait

1. Build the expert dependency graph (analyst nodes -> engineer node)
2. Calculate the friction score for each edge
3. What is the engineer's betweenness centrality?
4. What happens to throughput if the engineer is on leave for a week?
5. Propose two AI interventions ranked by estimated friction reduction

In [ ]:
# Exercise 2 -- ANSWER KEY
print('EXERCISE 2: Expert Dependency Network')
print('=' * 70)
print()
expert_handoffs = pd.DataFrame({
    'from_role':      ['Analyst_A','Analyst_B','Analyst_C','Analyst_A','Analyst_B','Analyst_A'],
    'to_role':        ['Engineer', 'Engineer', 'Engineer', 'Engineer', 'Engineer', 'Engineer'],
    'cases':          [10,          10,          10,          4,          6,          5],
    'avg_wait_hours': [4,           4,           4,           8,          8,          24],
    'description':    [
        'SQL query - A', 'SQL query - B', 'SQL query - C',
        'Dashboard fix - A', 'Dashboard fix - B',
        'Pipeline debug - A',
    ],
})
G_expert = build_process_graph(expert_handoffs)
print('Q1-2 -- Graph edges and friction scores:')
print(top_friction_edges(G_expert, top_n=10).to_string(index=False))
print()
expert_btw = nx.betweenness_centrality(G_expert, weight='avg_wait_hours', normalized=True)
print('Q3 -- Betweenness centrality:')
for node, score in sorted(expert_btw.items(), key=lambda x: x[1], reverse=True)[:4]:
    print(f'  {node}: {score:.4f}')
print()
total_m = expert_handoffs['cases'].sum()
print(f'Q4 -- If engineer on leave 5 days:')
print(f'  {total_m} requests/month cannot progress')
print(f'  Backlog builds at ~{total_m // 4} requests/week')
print('  All 3 analysts blocked -- entire BI workflow stalls')
print()
print('Q5 -- AI interventions ranked by friction reduction:')
print()
print('  1. Text-to-SQL assistant (Copilot-style)')
print('     Addresses: 30 SQL requests x 4 h = 120 friction units')
print('     Expected: 80% self-served -> removes ~96 friction units')
print()
print('  2. Automated dashboard refresh + anomaly alerts')
print('     Addresses: 10 requests x 8 h = 80 friction units')
print('     Expected: 60% auto-resolved -> removes ~48 friction units')

---

### Exercise 3: Capstone — Build a Business Case from Graph Evidence

Using the support triage graph from Part 4, build a one-page AI business case:

1. Identify the top two bottleneck nodes and their centrality scores
2. Attach L06 friction cost to each node (escalation EUR 6,075, rework EUR 3,528)
3. Propose one AI solution per node with estimated centrality reduction
4. Calculate expected annual savings if Q1 figures are annualised (x4)
5. Prioritise: fix rework first or escalation first? Justify with graph + cost evidence

In [ ]:
# Exercise 3 -- ANSWER KEY
print('EXERCISE 3: Support Triage Business Case')
print('=' * 80)
print()
print('STEP 1 -- Top bottleneck nodes:')
print(support_bottlenecks[['node','betweenness','total_cases_in']].head(2).to_string(index=False))
print()
print('STEP 2 -- L06 friction cost per node:')
print('  Analyst (rework loop):    EUR 3,528  (8% x 360 cases x 18 min x EUR 55/hr)')
print('  Specialist (escalation):  EUR 6,075  (15% x 675 cases x 30 min x EUR 75/hr)')
print()
print('STEP 3 -- AI interventions:')
print()
print('  Node 1: Analyst rework loop')
print('    Solution: LLM intent classifier on ticket submission')
print('    Effect:   Cut misroute 8% -> 2%, remove 75% of rework edge volume')
print()
print('  Node 2: Specialist escalation')
print('    Solution: LLM assisted resolution + specialist knowledge base')
print('    Effect:   Cut escalation 15% -> 5%, remove 67% of escalation edge')
print()
rework_q1     = 3528 * 0.75
escalation_q1 = 6075 * 0.67
total_annual  = (rework_q1 + escalation_q1) * 4
print('STEP 4 -- Expected savings (annualised):')
print(f'  Rework reduction:     EUR {rework_q1*4:,.0f}/year  (75% of EUR {3528*4:,})')
print(f'  Escalation reduction: EUR {escalation_q1*4:,.0f}/year  (67% of EUR {6075*4:,})')
print(f'  Total expected saving: EUR {total_annual:,.0f}/year')
print()
print('STEP 5 -- Priority recommendation:')
print()
print('  Fix ESCALATION first (specialist node):')
print(f'  -> Higher absolute saving (EUR {escalation_q1*4:,.0f} vs EUR {rework_q1*4:,.0f} per year)')
print('  -> Specialist edge has highest single friction score in the graph')
print('  -> L07: each escalation adds 6.1 days to lead time')
print('  -> LLM knowledge base built incrementally from specialist resolutions')
print()
print('  Then fix REWORK (analyst loop):')
print('  -> LLM classifier needs ticket labels -- collect during escalation fix')
print('  -> Rework savings fund the second implementation')

---

## Part 8 — Completion Checklist

**You have completed Lesson 08 when you can:**

- [ ] Build a directed weighted process graph from a handoff table using networkx
- [ ] Calculate betweenness and degree centrality and explain what each measures
- [ ] Identify the top bottleneck node and the top friction edge in any process graph
- [ ] Answer the six graph questions (hidden bottleneck, slowest path, single point of failure, etc.)
- [ ] Attach L06 friction costs (EUR) to graph nodes and edges
- [ ] Propose a targeted AI intervention for each bottleneck node
- [ ] Rank interventions by friction reduction and write a concise business case
- [ ] Complete Exercises 1, 2, and 3

---

## Key Insights Summary

**Lesson 08: Graph Bottleneck Detection**

**Betweenness centrality identifies structural bottlenecks** -- not just nodes that are slow,
but nodes that are unavoidable on many case paths.

**The highest-friction edge is often an exception path, not the happy path** --
Invoice escalation handles 2% of cases but creates a 20 h queue;
Support specialist handles 15% but accounts for the largest friction edge.

**Graph evidence + L06 friction cost = investment thesis** --
centrality tells you structural importance, friction EUR tells you financial value;
together they justify AI spend.

**Three graph types cover most enterprise workflows:**
- Person-task graph: who does what (approval flows, escalation networks)
- Document-entity graph: which artefacts are reused across workflows
- System dependency graph: which systems create manual bridges

**The recommended AI action follows from the graph:**
- High-centrality approver: rule-based or LLM auto-approval
- High-friction escalation edge: LLM classifier to reduce escalation rate
- Rework loop: data quality automation to eliminate rework at source